# COVID-19 Tweet Classification Challenge
**Objective**: Predict the sentiment/category of COVID-19 related tweets.
**Metric**: Multi-class Log Loss or Accuracy

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration

In [ ]:
def get_data_path():
    if os.path.exists('/kaggle/input'):
        for dirname, _, filenames in os.walk('/kaggle/input'):
            if 'Train.csv' in filenames:
                return os.path.join(dirname, 'Train.csv')
    return 'Train.csv'

TRAIN_CSV = get_data_path()
print(f"Using data from: {TRAIN_CSV}")

## 2. Dataset Class

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

## 3. Training and Evaluation Functions

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    correct_predictions = 0
    
    for d in tqdm(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        
        loss = outputs.loss
        logits = outputs.logits
        
        _, preds = torch.max(logits, dim=1)
        correct_predictions += torch.sum(preds == labels)
        losses.append(loss.item())
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    losses = []
    correct_predictions = 0
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            _, preds = torch.max(logits, dim=1)
            correct_predictions += torch.sum(preds == labels)
            losses.append(loss.item())
            
    return correct_predictions.double() / len(data_loader.dataset), np.mean(losses)

## 4. Main Execution

In [ ]:
if os.path.exists(TRAIN_CSV):
    train = pd.read_csv(TRAIN_CSV)
    # Assuming columns 'text' and 'target'
    if 'target' in train.columns:
        if train['target'].dtype == 'object':
            from sklearn.preprocessing import LabelEncoder
            le = LabelEncoder()
            train['target'] = le.fit_transform(train['target'])
            num_classes = len(le.classes_)
        else:
            num_classes = train['target'].nunique()
    else:
        print("Target column not found. Defaulting to 3 classes.")
        num_classes = 3

    MODEL_NAME = 'bert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)
    
    train_ds = TweetDataset(train_df.text.to_numpy(), train_df.target.to_numpy(), tokenizer)
    val_ds = TweetDataset(val_df.text.to_numpy(), val_df.target.to_numpy(), tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, correct_bias=False)
    
    EPOCHS = 3
    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        train_acc, train_loss = train_epoch(model, train_loader, optimizer, device)
        print(f'Train loss: {train_loss:.4f}, accuracy: {train_acc:.4f}')
        
        val_acc, val_loss = eval_model(model, val_loader, device)
        print(f'Val loss: {val_loss:.4f}, accuracy: {val_acc:.4f}')
else:
    print("Train.csv not found. Please attach the dataset on Kaggle.")